In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，创建输出目录
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import shap
import pickle
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

Path('results/feature_importance').mkdir(parents=True, exist_ok=True)


# =============================================================================
# 加载数据与模型
# =============================================================================

# 读取数据集
df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']

X = df[feature_cols].values
y = df[target_col].values

# 从各模型目录加载最终模型
def load_model(model_name):
    model_file = f'models/{model_name}/{model_name}_final_model.pkl'
    with open(model_file, 'rb') as f:
        pkg = pickle.load(f)
    return pkg['model'], pkg['scaler']

models_dict  = {}
scalers_dict = {}

for name in ['ridge', 'svr', 'rf', 'xgboost']:
    label = {'ridge': 'Ridge', 'svr': 'SVR', 'rf': 'RF', 'xgboost': 'XGB'}[name]
    try:
        models_dict[label], scalers_dict[label] = load_model(name)
    except Exception as e:
        print(f"{label} 加载失败: {e}")

# 加载DNN
try:
    from tensorflow import keras
    dnn_model = None
    for path in ['models/dnn/dnn_model.keras', 'models/dnn/dnn_final_model.keras']:
        if Path(path).exists():
            dnn_model = keras.models.load_model(path)
            break
    if dnn_model is None:
        raise FileNotFoundError("未找到DNN模型文件")
    models_dict['DNN'] = dnn_model

    scaler_loaded = False
    for path in ['models/dnn/dnn_final_model.pkl', 'models/dnn/dnn_model_package.pkl']:
        if Path(path).exists():
            with open(path, 'rb') as f:
                scalers_dict['DNN'] = pickle.load(f)['scaler']
            scaler_loaded = True
            break
    if not scaler_loaded:
        scaler_dnn = StandardScaler()
        scaler_dnn.fit(X)
        scalers_dict['DNN'] = scaler_dnn
except Exception as e:
    print(f"DNN 加载失败: {e}")




In [2]:
# =============================================================================
# SHAP分析（RF、XGB使用全量数据，Ridge使用小样本）
# =============================================================================

np.random.seed(42)

X_sample = X
y_sample = y

# Ridge使用KernelExplainer，计算量大，采用小样本
SMALL_SAMPLE    = min(5000, len(X))
small_idx       = np.random.choice(len(X_sample), SMALL_SAMPLE, replace=False)
X_small         = X_sample[small_idx]

np.save('results/feature_importance/sample_indices_large.npy', np.arange(len(X)))
np.save('results/feature_importance/sample_indices_small.npy', small_idx)

# --- RF ---
scaler_rf        = scalers_dict['RF']
X_scaled_rf      = scaler_rf.transform(X_sample)
explainer_rf     = shap.TreeExplainer(models_dict['RF'])
shap_values_rf   = explainer_rf.shap_values(X_scaled_rf)

np.save('results/feature_importance/shap_values_rf.npy', shap_values_rf)

df_shap_rf = pd.DataFrame({
    'feature':          feature_cols,
    'importance_mean':  np.abs(shap_values_rf).mean(axis=0),
    'importance_std':   np.abs(shap_values_rf).std(axis=0)
}).sort_values('importance_mean', ascending=False)
df_shap_rf.to_csv('results/feature_importance/shap_importance_rf.csv', index=False)

# --- XGB ---
scaler_xgb       = scalers_dict['XGB']
X_scaled_xgb     = scaler_xgb.transform(X_sample)
explainer_xgb    = shap.TreeExplainer(models_dict['XGB'])
shap_values_xgb  = explainer_xgb.shap_values(X_scaled_xgb)

np.save('results/feature_importance/shap_values_xgb.npy', shap_values_xgb)

df_shap_xgb = pd.DataFrame({
    'feature':          feature_cols,
    'importance_mean':  np.abs(shap_values_xgb).mean(axis=0),
    'importance_std':   np.abs(shap_values_xgb).std(axis=0)
}).sort_values('importance_mean', ascending=False)
df_shap_xgb.to_csv('results/feature_importance/shap_importance_xgb.csv', index=False)

# --- Ridge（KernelExplainer，取100个样本计算）---
scaler_ridge         = scalers_dict['Ridge']
X_small_scaled_ridge = scaler_ridge.transform(X_small)
background_ridge     = shap.sample(X_small_scaled_ridge, 100)
explainer_ridge      = shap.KernelExplainer(models_dict['Ridge'].predict, background_ridge)
shap_values_ridge    = explainer_ridge.shap_values(X_small_scaled_ridge[:100])

np.save('results/feature_importance/shap_values_ridge.npy', shap_values_ridge)

df_shap_ridge = pd.DataFrame({
    'feature':          feature_cols,
    'importance_mean':  np.abs(shap_values_ridge).mean(axis=0),
    'importance_std':   np.abs(shap_values_ridge).std(axis=0)
}).sort_values('importance_mean', ascending=False)
df_shap_ridge.to_csv('results/feature_importance/shap_importance_ridge.csv', index=False)




  0%|          | 0/100 [00:00<?, ?it/s]

In [3]:
# =============================================================================
# Permutation Importance（所有模型）
# =============================================================================

perm_importance_results = {}

# Ridge、SVR、RF、XGB
for model_name in ['Ridge', 'SVR', 'RF', 'XGB']:
    if model_name not in models_dict:
        continue
    X_scaled = scalers_dict[model_name].transform(X_sample)
    perm_res  = permutation_importance(
        models_dict[model_name], X_scaled, y_sample,
        n_repeats=10, random_state=42, n_jobs=-1
    )
    perm_importance_results[model_name] = {
        'importances_mean': perm_res.importances_mean.tolist(),
        'importances_std':  perm_res.importances_std.tolist()
    }

# DNN（不支持并行）
if 'DNN' in models_dict:
    X_scaled_dnn = scalers_dict['DNN'].transform(X_sample)

    def dnn_scorer(model, X, y):
        y_pred = model.predict(X, verbose=0).flatten()
        return -np.mean((y - y_pred) ** 2)

    perm_res_dnn = permutation_importance(
        models_dict['DNN'], X_scaled_dnn, y_sample,
        n_repeats=10, random_state=42,
        scoring=dnn_scorer, n_jobs=1
    )
    perm_importance_results['DNN'] = {
        'importances_mean': perm_res_dnn.importances_mean.tolist(),
        'importances_std':  perm_res_dnn.importances_std.tolist()
    }

with open('results/feature_importance/perm_importance_all.json', 'w') as f:
    json.dump(perm_importance_results, f, indent=4)

for model_name, res in perm_importance_results.items():
    pd.DataFrame({
        'feature':          feature_cols,
        'importance_mean':  res['importances_mean'],
        'importance_std':   res['importances_std']
    }).sort_values('importance_mean', ascending=False).to_csv(
        f'results/feature_importance/perm_importance_{model_name.lower()}.csv', index=False
    )




In [4]:
# =============================================================================
# 原生特征重要性（RF Gini，XGB Gain）
# =============================================================================

native_importance = {}

# RF Gini
if 'RF' in models_dict:
    rf_imp = models_dict['RF'].feature_importances_
    pd.DataFrame({
        'feature':    feature_cols,
        'importance': rf_imp
    }).sort_values('importance', ascending=False).to_csv(
        'results/feature_importance/native_importance_rf.csv', index=False
    )
    native_importance['RF'] = rf_imp.tolist()

# XGB Gain
if 'XGB' in models_dict:
    xgb_score = models_dict['XGB'].get_booster().get_score(importance_type='gain')
    xgb_imp   = [xgb_score.get(f'f{i}', 0.0) for i in range(len(feature_cols))]
    pd.DataFrame({
        'feature':          feature_cols,
        'importance_gain':  xgb_imp
    }).sort_values('importance_gain', ascending=False).to_csv(
        'results/feature_importance/native_importance_xgb.csv', index=False
    )
    native_importance['XGB'] = xgb_imp

with open('results/feature_importance/native_importance_all.json', 'w') as f:
    json.dump(native_importance, f, indent=4)